In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler

# Çıktı ekranında gereksiz uyarı mesajlarını gizlemek için
warnings.filterwarnings('ignore')


In [33]:
# Verilerin bulunduğu klasör yolu
data_path = "../data/raw/"

# CSV dosyalarını Pandas DataFrame olarak okuma
student_info = pd.read_csv(os.path.join(data_path, "studentInfo.csv"))
student_assessment = pd.read_csv(os.path.join(data_path, "studentAssessment.csv"))
student_vle = pd.read_csv(os.path.join(data_path, "studentVle.csv"))
assessments = pd.read_csv(os.path.join(data_path, "assessments.csv"))
courses = pd.read_csv(os.path.join(data_path, "courses.csv"))
student_registration = pd.read_csv(os.path.join(data_path, "studentRegistration.csv"))
vle = pd.read_csv(os.path.join(data_path, "vle.csv"))

In [34]:
# Tüm DataFrame'leri bir sözlükte toplama (döngü ile kolayca yazdırmak için)
dataframes = {
    "student_info": student_info,
    "student_assessment": student_assessment,
    "student_vle": student_vle,
    "assessments": assessments,
    "courses": courses,
    "student_registration": student_registration,
    "vle": vle
}

# Her bir tablonun shape ve kolon bilgilerini yazdırma
print("--- OULAD VERİ SETİ TABLO BİLGİLERİ ---\n")
for name, df in dataframes.items():
    print(f"Tablo Adı: {name}")
    print(f"Boyut (Satır, Sütun): {df.shape}")
    print(f"Sütunlar: {list(df.columns)}")
    print("-" * 50)


--- OULAD VERİ SETİ TABLO BİLGİLERİ ---

Tablo Adı: student_info
Boyut (Satır, Sütun): (32593, 12)
Sütunlar: ['code_module', 'code_presentation', 'id_student', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result']
--------------------------------------------------
Tablo Adı: student_assessment
Boyut (Satır, Sütun): (173912, 5)
Sütunlar: ['id_assessment', 'id_student', 'date_submitted', 'is_banked', 'score']
--------------------------------------------------
Tablo Adı: student_vle
Boyut (Satır, Sütun): (10655280, 6)
Sütunlar: ['code_module', 'code_presentation', 'id_student', 'id_site', 'date', 'sum_click']
--------------------------------------------------
Tablo Adı: assessments
Boyut (Satır, Sütun): (206, 6)
Sütunlar: ['code_module', 'code_presentation', 'id_assessment', 'assessment_type', 'date', 'weight']
--------------------------------------------------
Tablo Adı: courses
Boyut (Satır, Sütun): (22,

In [35]:
# İşlem öncesi eksik değer sayısını kontrol edelim
print("student_info - İşlem ÖNCESİ 'imd_band' eksik sayısı:", student_info['imd_band'].isnull().sum())

# En sık tekrar eden (mod) değeri bulup eksikleri bu değerle dolduralım
imd_band_mode = student_info['imd_band'].mode()[0]
student_info['imd_band'].fillna(imd_band_mode, inplace=True)

# İşlem sonrası eksik değer sayısını kontrol edelim
print("student_info - İşlem SONRASI 'imd_band' eksik sayısı:", student_info['imd_band'].isnull().sum())


student_info - İşlem ÖNCESİ 'imd_band' eksik sayısı: 1111
student_info - İşlem SONRASI 'imd_band' eksik sayısı: 0


In [36]:
# İşlem öncesi eksik değer sayısını kontrol edelim
print("student_assessment - İşlem ÖNCESİ 'score' eksik sayısı:", student_assessment['score'].isnull().sum())

# Eksikleri medyan ile dolduralım
score_median = student_assessment['score'].median()
student_assessment['score'].fillna(score_median, inplace=True)

# İşlem sonrası eksik değer sayısını kontrol edelim
print("student_assessment - İşlem SONRASI 'score' eksik sayısı:", student_assessment['score'].isnull().sum())


student_assessment - İşlem ÖNCESİ 'score' eksik sayısı: 173
student_assessment - İşlem SONRASI 'score' eksik sayısı: 0


In [37]:
def handle_other_missing_values(df, df_name):
    print(f"\n--- {df_name} İçin Diğer Sütunların Kontrolü ---")
    total_rows = len(df)

    for col in df.columns:
        missing_count = df[col].isnull().sum()
        if missing_count > 0:
            missing_percentage = (missing_count / total_rows) * 100

            if missing_percentage <= 5.0:
                print(f"{col} sütununda %{missing_percentage:.2f} eksik var. (<= %5). Mod ile dolduruluyor.")
                mode_val = df[col].mode()[0]
                df[col].fillna(mode_val, inplace=True)
            else:
                print(f"DİKKAT: {col} sütununda %{missing_percentage:.2f} eksik var. (> %5). Raporlandı, otomatik doldurulmadı.")

# İki ana DataFrame'imiz için fonksiyonu çağıralım
handle_other_missing_values(student_info, "student_info")
handle_other_missing_values(student_assessment, "student_assessment")



--- student_info İçin Diğer Sütunların Kontrolü ---

--- student_assessment İçin Diğer Sütunların Kontrolü ---


In [38]:
print("\n--- SON KONTROL ---")
info_missing = student_info.isnull().sum().sum()
assessment_missing = student_assessment.isnull().sum().sum()

print(f"student_info tablosunda kalan toplam eksik değer: {info_missing}")
if info_missing == 0:
    print("✓ student_info eksiksiz ve temiz!")

print(f"student_assessment tablosunda kalan toplam eksik değer: {assessment_missing}")
if assessment_missing == 0:
    print("✓ student_assessment eksiksiz ve temiz!")



--- SON KONTROL ---
student_info tablosunda kalan toplam eksik değer: 0
✓ student_info eksiksiz ve temiz!
student_assessment tablosunda kalan toplam eksik değer: 0
✓ student_assessment eksiksiz ve temiz!


In [39]:
# Cinsiyet (gender) sütunu için eşleştirme: M -> 0, F -> 1
student_info['gender'] = student_info['gender'].map({'M': 0, 'F': 1})

# Engel durumu (disability) sütunu için eşleştirme: Y -> 0, N -> 1
# (Eğer Y'nin 1, N'nin 0 olmasını isterseniz sözlüğü {'Y': 1, 'N': 0} olarak değiştirebilirsiniz)
student_info['disability'] = student_info['disability'].map({'Y': 0, 'N': 1})

print("Label Encoding işlemi tamamlandı.")


Label Encoding işlemi tamamlandı.


In [40]:
# One-Hot Encoding uygulanacak sütunların listesi
categorical_cols = ['region', 'highest_education', 'age_band', 'imd_band']

# get_dummies ile One-Hot Encoding uygulanması
# dtype=int diyerek True/False yerine 1/0 olarak yazılmasını sağlıyoruz
student_info = pd.get_dummies(student_info, columns=categorical_cols, drop_first=True, dtype=int)

print("One-Hot Encoding işlemi tamamlandı.")


One-Hot Encoding işlemi tamamlandı.


In [41]:
# Tüm kodlama işlemleri sonrası oluşan yeni sütun isimlerini yazdırma
print("\n--- Yeni Sütun İsimleri ---")
for col in student_info.columns:
    print(col)

# student_info DataFrame'inin yeni boyutunu (shape) gösterme
print(f"\n--- student_info Yeni Boyut ---")
print(f"Satır Sayısı: {student_info.shape[0]}")
print(f"Sütun Sayısı: {student_info.shape[1]}")



--- Yeni Sütun İsimleri ---
code_module
code_presentation
id_student
gender
num_of_prev_attempts
studied_credits
disability
final_result
region_East Midlands Region
region_Ireland
region_London Region
region_North Region
region_North Western Region
region_Scotland
region_South East Region
region_South Region
region_South West Region
region_Wales
region_West Midlands Region
region_Yorkshire Region
highest_education_HE Qualification
highest_education_Lower Than A Level
highest_education_No Formal quals
highest_education_Post Graduate Qualification
age_band_35-55
age_band_55<=
imd_band_10-20
imd_band_20-30%
imd_band_30-40%
imd_band_40-50%
imd_band_50-60%
imd_band_60-70%
imd_band_70-80%
imd_band_80-90%
imd_band_90-100%

--- student_info Yeni Boyut ---
Satır Sayısı: 32593
Sütun Sayısı: 35


In [42]:
import numpy as np

# Eğer date <= 28 ise o günkü sum_click değerini al, değilse 0 yaz.
student_vle['early_clicks_temp'] = np.where(student_vle['date'] <= 28, student_vle['sum_click'], 0)


In [43]:
# Belirtilen 3'lü seviyede gruplama ve aggregasyon işlemleri
vle_features = student_vle.groupby(['id_student', 'code_module', 'code_presentation']).agg(
    sum_clicks=('sum_click', 'sum'),           # Toplam tıklama sayısı
    active_days=('date', 'nunique'),           # Kaç farklı günde sisteme girdiği
    first_interaction=('date', 'min'),         # İlk etkileşim günü (min date)
    last_interaction=('date', 'max'),          # Son etkileşim günü (max date)
    early_clicks=('early_clicks_temp', 'sum')  # İlk 28 gündeki toplam tıklamalar
).reset_index() # Gruplanan indeksleri tekrar standart sütuna çevirir

# İşimiz biten geçici sütunu ana tablodan (student_vle) silelim ki hafızada yer kaplamasın
student_vle.drop(columns=['early_clicks_temp'], inplace=True)


In [44]:
print("--- vle_features Tablosu Bilgileri ---")
print(f"Oluşturulan tablonun boyutu (Satır, Sütun): {vle_features.shape}")

# Yeni oluşturduğumuz özellikleri kontrol etmek için ilk 5 satırı yazdıralım
vle_features.head()


--- vle_features Tablosu Bilgileri ---
Oluşturulan tablonun boyutu (Satır, Sütun): (29228, 8)


,id_student,code_module,code_presentation,sum_clicks,active_days,first_interaction,last_interaction,early_clicks
0,6516,AAA,2014J,2791,159,-23,269,826
1,8462,DDD,2013J,646,56,-6,118,375
2,8462,DDD,2014J,10,1,10,10,10
3,11391,AAA,2013J,934,40,-5,253,401
4,23629,BBB,2013B,161,16,-6,87,53


In [45]:
import pandas as pd
import numpy as np

# Her iki tabloyu id_assessment üzerinden 'left join' mantığıyla birleştiriyoruz
merged_assessments = pd.merge(student_assessment, assessments, on='id_assessment', how='left')


In [46]:
# Eğer öğrencinin teslim günü (date_submitted), sınavın son tarihinden (date) büyükse 1, değilse 0
merged_assessments['is_late'] = np.where(merged_assessments['date_submitted'] > merged_assessments['date'], 1, 0)

# Ağırlıklı ortalamayı hesaplamak için öncelikle (Skor * Ağırlık) değerini buluyoruz
merged_assessments['weighted_score_temp'] = merged_assessments['score'] * merged_assessments['weight']


In [47]:
# Öğrenci, ders ve dönem bazında gruplama işlemleri
assessment_features = merged_assessments.groupby(['id_student', 'code_module', 'code_presentation']).agg(
    avg_score=('score', 'mean'),                           # Ortalama skor
    submission_count=('id_assessment', 'count'),           # Tamamlanan ödev sayısı
    late_submission_count=('is_late', 'sum'),              # Geç teslim sayısı
    sum_weighted_score=('weighted_score_temp', 'sum'),     # (Skor * Ağırlık) toplamı
    sum_weight=('weight', 'sum')                           # Toplam ağırlık
).reset_index()


In [48]:
# Geç teslim oranının hesaplanması (Geç Teslim Sayısı / Toplam Teslim Sayısı)
assessment_features['late_submission_rate'] = assessment_features['late_submission_count'] / assessment_features['submission_count']

# Ağırlıklı ortalamanın hesaplanması.
# Eğer toplam ağırlık 0'dan büyükse bölme işlemini yap, ağırlık 0 ise 0 değeri ata (Sıfıra bölme hatasını engellemek için)
assessment_features['weighted_avg_score'] = np.where(
    assessment_features['sum_weight'] > 0,
    assessment_features['sum_weighted_score'] / assessment_features['sum_weight'],
    0
)

# Hesaplamada kullandığımız geçici sütunları temizliyoruz
assessment_features.drop(columns=['sum_weighted_score', 'sum_weight'], inplace=True)


In [49]:
print("--- assessment_features Tablosu Bilgileri ---")
print(f"Oluşturulan tablonun boyutu (Satır, Sütun): {assessment_features.shape}")

# Sonuçları gözlemlemek için ilk 5 satırı yazdırıyoruz
assessment_features.head()


--- assessment_features Tablosu Bilgileri ---
Oluşturulan tablonun boyutu (Satır, Sütun): (25843, 8)


,id_student,code_module,code_presentation,avg_score,submission_count,late_submission_count,late_submission_rate,weighted_avg_score
0,6516,AAA,2014J,61.800000,5,0,0.000000,63.50
1,8462,DDD,2013J,87.666667,3,1,0.333333,87.25
2,8462,DDD,2014J,86.500000,4,0,0.000000,86.00
3,11391,AAA,2013J,82.000000,5,0,0.000000,82.40
4,23629,BBB,2013B,82.500000,4,3,0.750000,66.76


In [50]:
# Birleştirme işlemlerinde kullanılacak ortak anahtar (Key) sütunları
merge_keys = ['id_student', 'code_module', 'code_presentation']

# student_info ile vle_features tablolarının birleştirilmesi
master_df = pd.merge(student_info, vle_features, on=merge_keys, how='left')


In [51]:
# assessment_features tablosunun master_df üzerine eklenmesi
master_df = pd.merge(master_df, assessment_features, on=merge_keys, how='left')


In [52]:
# Yalnızca anahtar sütunları ve ihtiyacımız olan 2 tarih sütununu seçiyoruz
reg_cols = merge_keys + ['date_registration', 'date_unregistration']

# student_registration verilerinin master_df üzerine eklenmesi
master_df = pd.merge(master_df, student_registration[reg_cols], on=merge_keys, how='left')


In [53]:
# Birleştirme sonrası tablonun boyutunu yazdırıyoruz
print(f"Birleştirme Sonrası master_df Boyutu: {master_df.shape}")

# Kalan tüm eksik değerleri (NaN) 0 ile dolduruyoruz
master_df.fillna(0, inplace=True)
print("Eksik değerler (NaN) başarıyla 0 ile dolduruldu.")

# 'id_student' HARİCİNDE isminde 'id' geçen gereksiz sütunları tespit edip siliyoruz.
# Not: Eğer code_module ve code_presentation sütunlarını da silmek isterseniz bu listeye manuel ekleyebilirsiniz.
cols_to_drop = [col for col in master_df.columns if 'id_' in col and col != 'id_student']

if len(cols_to_drop) > 0:
    master_df.drop(columns=cols_to_drop, inplace=True)
    print(f"Silinen gereksiz id sütunları: {cols_to_drop}")

print("\n--- Nihai master_df Bilgileri ---")
print(f"Son Boyut: {master_df.shape}")
master_df.head()


Birleştirme Sonrası master_df Boyutu: (32593, 47)
Eksik değerler (NaN) başarıyla 0 ile dolduruldu.

--- Nihai master_df Bilgileri ---
Son Boyut: (32593, 47)


,code_module,code_presentation,id_student,gender,num_of_prev_attempts,studied_credits,disability,final_result,region_East Midlands Region,region_Ireland,...,first_interaction,last_interaction,early_clicks,avg_score,submission_count,late_submission_count,late_submission_rate,weighted_avg_score,date_registration,date_unregistration
0,AAA,2013J,11391,0,0,240,1,Pass,0,0,...,-5.0,253.0,401.0,82.0,5.0,0.0,0.0,82.4,-159.0,0.0
1,AAA,2013J,28400,1,0,60,1,Pass,0,0,...,-10.0,239.0,618.0,66.4,5.0,2.0,0.4,65.4,-53.0,0.0
2,AAA,2013J,30268,1,0,60,0,Withdrawn,0,0,...,-10.0,12.0,281.0,0.0,0.0,0.0,0.0,0.0,-92.0,12.0
3,AAA,2013J,31604,1,0,60,1,Pass,0,0,...,-10.0,264.0,494.0,76.0,5.0,0.0,0.0,76.3,-52.0,0.0
4,AAA,2013J,32885,1,0,60,1,Pass,0,0,...,-10.0,247.0,567.0,54.4,5.0,5.0,1.0,55.0,-176.0,0.0


In [54]:
# Hedef değişkendeki sınıfların sayısal karşılıklarını bir sözlük (dictionary) olarak tanımlıyoruz
target_mapping = {
    'Pass': 0,
    'Fail': 1,
    'Distinction': 2,
    'Withdrawn': 3
}

# Yeni 'target' sütununu oluşturuyoruz
master_df['target'] = master_df['final_result'].map(target_mapping)

print("Hedef değişken (target) başarıyla oluşturuldu.")


Hedef değişken (target) başarıyla oluşturuldu.


In [55]:
from sklearn.preprocessing import StandardScaler

# Tüm sayısal veri tiplerine sahip sütunları bulalım
numerical_cols = master_df.select_dtypes(include=['int64', 'float64', 'int32', 'float32']).columns.tolist()

# Ölçeklenmemesi gereken sütunlar (Öğrenci ID'si ve yeni oluşturduğumuz hedef değişken)
exclude_cols = ['id_student', 'target']

# İşlem uygulanacak nihai sütun listesini oluşturuyoruz
cols_to_scale = [col for col in numerical_cols if col not in exclude_cols]

# Sütunları ölçekleme (Standardization) işlemi
scaler = StandardScaler()
master_df[cols_to_scale] = scaler.fit_transform(master_df[cols_to_scale])

print("Ölçeklenen Sütunlar:\n", cols_to_scale)


Ölçeklenen Sütunlar:
 ['gender', 'num_of_prev_attempts', 'studied_credits', 'disability', 'region_East Midlands Region', 'region_Ireland', 'region_London Region', 'region_North Region', 'region_North Western Region', 'region_Scotland', 'region_South East Region', 'region_South Region', 'region_South West Region', 'region_Wales', 'region_West Midlands Region', 'region_Yorkshire Region', 'highest_education_HE Qualification', 'highest_education_Lower Than A Level', 'highest_education_No Formal quals', 'highest_education_Post Graduate Qualification', 'age_band_35-55', 'age_band_55<=', 'imd_band_10-20', 'imd_band_20-30%', 'imd_band_30-40%', 'imd_band_40-50%', 'imd_band_50-60%', 'imd_band_60-70%', 'imd_band_70-80%', 'imd_band_80-90%', 'imd_band_90-100%', 'sum_clicks', 'active_days', 'first_interaction', 'last_interaction', 'early_clicks', 'avg_score', 'submission_count', 'late_submission_count', 'late_submission_rate', 'weighted_avg_score', 'date_registration', 'date_unregistration']


In [56]:
print("--- master_df Son Durum Kontrolü ---")
print(f"Toplam Satır Sayısı: {master_df.shape[0]}")
print(f"Toplam Sütun Sayısı: {master_df.shape[1]}")

print("\n--- Sütun Listesi ---")
print(list(master_df.columns))

print("\n--- İlk 5 Satır Görünümü ---")
master_df.head()


--- master_df Son Durum Kontrolü ---
Toplam Satır Sayısı: 32593
Toplam Sütun Sayısı: 48

--- Sütun Listesi ---
['code_module', 'code_presentation', 'id_student', 'gender', 'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result', 'region_East Midlands Region', 'region_Ireland', 'region_London Region', 'region_North Region', 'region_North Western Region', 'region_Scotland', 'region_South East Region', 'region_South Region', 'region_South West Region', 'region_Wales', 'region_West Midlands Region', 'region_Yorkshire Region', 'highest_education_HE Qualification', 'highest_education_Lower Than A Level', 'highest_education_No Formal quals', 'highest_education_Post Graduate Qualification', 'age_band_35-55', 'age_band_55<=', 'imd_band_10-20', 'imd_band_20-30%', 'imd_band_30-40%', 'imd_band_40-50%', 'imd_band_50-60%', 'imd_band_60-70%', 'imd_band_70-80%', 'imd_band_80-90%', 'imd_band_90-100%', 'sum_clicks', 'active_days', 'first_interaction', 'last_interaction', 'early_clicks', 

,code_module,code_presentation,id_student,gender,num_of_prev_attempts,studied_credits,disability,final_result,region_East Midlands Region,region_Ireland,...,last_interaction,early_clicks,avg_score,submission_count,late_submission_count,late_submission_rate,weighted_avg_score,date_registration,date_unregistration,target
0,AAA,2013J,11391,-0.907405,-0.340229,3.901543,0.327892,Pass,-0.279712,-0.194155,...,0.930961,0.220302,0.738145,-0.077641,-0.689131,-0.764243,0.935066,-1.819411,-0.299843,0
1,AAA,2013J,28400,1.102043,-0.340229,-0.481083,0.327892,Pass,-0.279712,-0.194155,...,0.792323,0.743853,0.263842,-0.077641,0.221727,0.527646,0.439842,0.330988,-0.299843,0
2,AAA,2013J,30268,1.102043,-0.340229,-0.481083,-3.049787,Withdrawn,-0.279712,-0.194155,...,-1.455607,-0.069220,-1.754985,-1.233458,-0.689131,-0.764243,-1.465314,-0.460196,-0.065839,3
3,AAA,2013J,31604,1.102043,-0.340229,-0.481083,0.327892,Pass,-0.279712,-0.194155,...,1.039892,0.444681,0.555721,-0.077641,-0.689131,-0.764243,0.757368,0.351275,-0.299843,0
4,AAA,2013J,32885,1.102043,-0.340229,-0.481083,0.327892,Pass,-0.279712,-0.194155,...,0.871545,0.620806,-0.101006,-0.077641,1.588014,2.465479,0.136881,-2.164286,-0.299843,0


In [57]:
import os

# Kayıt klasörünün yolunu belirtiyoruz, eğer bu klasör yoksa oluşturuyoruz
output_dir = '../data/processed/'
os.makedirs(output_dir, exist_ok=True)

# Dosyayı indeks sütunu olmadan CSV olarak kaydetme işlemi
file_path = os.path.join(output_dir, 'dataset_clean.csv')
master_df.to_csv(file_path, index=False)

# Dosyanın kaydedildiğini doğrulamak için sistemden boyutunu MB cinsinden okuyup yazdırıyoruz
file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
print(f"Nihai veri seti '{file_path}' yoluna başarıyla kaydedildi!")
print(f"Kaydedilen Dosya Boyutu: {file_size_mb:.2f} MB")

Nihai veri seti '../data/processed/dataset_clean.csv' yoluna başarıyla kaydedildi!
Kaydedilen Dosya Boyutu: 27.55 MB
